In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import box
import sys
import numpy as np
import networkx as nx

In [ ]:
sys.path.append('../src/nlr_ghe_screen/clustering/')
from distance_matrices import (
    construct_euclidian_dist_matrix,
    construct_road_network_dist_matrix,
)
from seed_clustering import (
    seed_clusters, 
    find_seed_centroids,
)
from construct_clusters import (
    build_clusters,
    find_nearby_buildings_network
)

sys.path.append('../src/nlr_ghe_screen/api/')
from building_footprint_api import (
    get_overture_footprints,
    get_ornl_footprints,
    join_ornl_overture_buildings
    
)
from osmnx_api import (
    get_road_network,
    plot_network_graph
)

In [ ]:
TEST_BBOX = (47.58, -122.36, 47.61, -122.31) # xmin, ymin, xmax, ymax
south, west, north, east = TEST_BBOX

In [ ]:
ghe_locations = gpd.read_file("../src/exports/ghe_locations.geojson")
geom_box = box(west, south, east, north)
cropped_ghes = gpd.clip(ghe_locations, geom_box)

In [ ]:
overture_buildings = get_overture_footprints(bbox=TEST_BBOX)
ornl_buildings = get_ornl_footprints(bbox=TEST_BBOX)

In [ ]:
cropped_ghes = cropped_ghes.to_crs(cropped_ghes.estimate_utm_crs())
ornl_buildings = ornl_buildings.to_crs(cropped_ghes.crs)
overture_buildings = overture_buildings.to_crs(cropped_ghes.crs)

In [ ]:
# network = get_road_network(bbox=TEST_BBOX)

In [ ]:
dist_matrix = construct_euclidian_dist_matrix(
    buildings_df=cropped_ghes,
    max_distance=500,
)

In [ ]:
buildings_joined = join_ornl_overture_buildings(
    ornl_data=ornl_buildings,
    overture_data=overture_buildings
)


In [ ]:
labels = seed_clusters(
    dist_matrix=dist_matrix,
    max_distance=500
)
cropped_ghes['cluster'] = labels

In [ ]:
centroids = find_seed_centroids(
    ghe_locations=cropped_ghes,
    cluster_col='cluster'
)

In [ ]:
clusters = build_clusters(
    seed_centroids=centroids,
    building_data=buildings_joined,
)

In [ ]:
c1 = cropped_ghes['cluster'].value_counts().rename("n1")
c2 = clusters['cluster'].value_counts().rename("n2")

counts = pd.concat([c1, c2], axis=1).fillna(0).astype(int)
counts["min"] = counts[["n1", "n2"]].min(axis=1)

ranked = counts.sort_values("min", ascending=False)      
ordered_labels = ranked.index.tolist()
ordered_labels[:10]

In [ ]:
idx = 1
m = centroids[centroids['cluster'] == idx].explore(color='red')
cropped_ghes[cropped_ghes['cluster'] == idx].explore(m=m, color='green')
clusters[clusters['cluster'] == idx].explore(m=m, color='blue')